# Schema Quality: nullable types, reasoning prompts & enums

Schema design, not just the model, decides how good your Retab extractions are.
This notebook proves it with a small, reproducible experiment and ends with a
linter that catches the problems automatically.

**What you'll see**

1. A corpus of five invoices of the *same* type that differ in which optional
   fields are printed and in *convention* (negative discount, European dates,
   verbose currency forms).
2. Four schemas for that one corpus: the **baseline** produced by
   `retab schemas generate`, then **nullable**, **reasoning** and **enum**
   upgrades.
3. A measured comparison (accuracy + consensus stability).
4. A static **linter** whose warnings line up with the measured failures.

## 1. The corpus

All five documents are invoices processed by one schema. They differ only in
*form* — which optional fields appear:

| Field | full | no_name | no_code | minimal | mixed |
|---|:--:|:--:|:--:|:--:|:--:|
| customer_name | ✓ | · | ✓ | · | ✓ |
| customer_code | ✓ | ✓ | · | · | ✓ |
| purchase_order | ✓ | ✓ | · | · | · |
| discount | ✓ | ✓ | ✓ | · | · |
| due_date | ✓ | ✓ | ✓ | · | ✓ |
| tax | ✓ | ✓ | ✓ | · | ✓ |

(✓ present, · absent). This is the shape that tests **nullable types**: a field
present on some invoices and genuinely absent on others. The corpus also varies
*convention* so more than one lever is tested: `discount` is printed negative,
several dates use European `DD/MM/YYYY`, and `currency` is printed in verbose
forms (`Euros`, `US Dollars`, `Pounds Sterling`, `€`) while ground truth is the
canonical ISO-4217 code. `documents/manifest.json` holds the correct value of
every field.

In [1]:
# Build the corpus PDFs (only if missing) and the schema variants.
import os, sys, subprocess
if not os.path.exists("documents/invoice_full.pdf"):
    subprocess.run([sys.executable, "generate_documents.py"], check=True)
from harness.variants import build_all
build_all()  # writes schemas/nullable.json and schemas/reasoning.json from baseline.json
print("corpus and schemas ready")

wrote C:\Users\DELL\Desktop\retab\cookbook\schema-quality\schemas\nullable.json
wrote C:\Users\DELL\Desktop\retab\cookbook\schema-quality\schemas\reasoning.json
wrote C:\Users\DELL\Desktop\retab\cookbook\schema-quality\schemas\enum.json
corpus and schemas ready


## 2. The four schemas

Each variant adds exactly one lever, so every difference is attributable.

- **baseline** — `retab schemas generate` from one invoice. Every field
  `required`, no nullable types, no reasoning prompts.
- **nullable** — the optional fields retyped `["<type>", "null"]`. They stay
  `required`: under strict structured output every property is emitted anyway,
  so optionality is carried by the null type, not by dropping the key.
- **reasoning** — nullable plus an **`X-ReasoningPrompt`** per optional field.
- **enum** — reasoning plus an **`enum`** of ISO-4217 codes on `currency`,
  normalizing the printed vocabulary (`Euros`, `US Dollars`, `€`) to a code.

**What is a reasoning prompt?** `X-ReasoningPrompt` is a Retab schema extension.
Its text is given to the model as a private, per-field "think about it this way"
instruction; it never appears in the output, it only steers extraction. Here it
says, per field, *"only return this if it is actually printed; if absent return
null; the discount is negative."* `X-SystemPrompt` is the same idea for the
whole schema.

Look at one field, `discount_amount`, across the schemas:

In [2]:
import json
def show(field):
    for name in ["baseline", "nullable", "reasoning", "enum"]:
        prop = json.load(open(f"schemas/{name}.json", encoding="utf-8"))["properties"][field]
        print(f"--- {name} ---")
        print("  type:", prop.get("type"), "| enum:", prop.get("enum"))
        if "X-ReasoningPrompt" in prop:
            print("  X-ReasoningPrompt:", prop["X-ReasoningPrompt"][:80] + "...")
show("discount_amount")
print()
show("currency")  # the enum lever only changes on the last variant

--- baseline ---
  type: number | enum: None
--- nullable ---
  type: ['number', 'null'] | enum: None
--- reasoning ---
  type: ['number', 'null'] | enum: None
  X-ReasoningPrompt: Return the discount only if a discount line is present (use the printed value, t...
--- enum ---
  type: ['number', 'null'] | enum: None
  X-ReasoningPrompt: Return the discount only if a discount line is present (use the printed value, t...

--- baseline ---
  type: string | enum: None
--- nullable ---
  type: string | enum: None
--- reasoning ---
  type: string | enum: None
--- enum ---
  type: string | enum: ['EUR', 'USD', 'GBP', 'CHF', 'JPY']


## 3. Run the experiment

For each (invoice, variant) we extract at `n_consensus=5` via the Retab CLI,
score against the manifest, and report. Results are cached under `results/`, so
re-running this cell is free.

In [3]:
from harness.experiment import run
from IPython.display import Markdown, display
display(Markdown(run(n_consensus=5, model="retab-micro")))

[invoice_full] baseline ...


[invoice_full] baseline acc=92% (cached)


[invoice_full] nullable ...


[invoice_full] nullable acc=92% (cached)


[invoice_full] reasoning ...


...

[invoice_mixed] reasoning ...


[invoice_mixed] reasoning acc=92% (cached)


[invoice_mixed] enum ...


[invoice_mixed] enum acc=100% (cached)


# Schema-Quality Experiment: nullable, reasoning & enum vs the generated baseline

Five invoices of the same type, processed by one schema. They differ in which optional fields are present (customer name, customer code, purchase order, discount, due date) and in *convention*: the discount is printed negative, several dates use European `DD/MM/YYYY`, and the currency is printed in verbose forms (`Euros`, `US Dollars`, `Pounds Sterling`, `€`) while ground truth is the ISO-4217 code. Four schema variants are compared, each adding one lever:

- **baseline** — produced by `retab schemas generate` from one invoice; every field `required`, no nullable types, no reasoning prompts.
- **nullable** — baseline with the optional fields retyped `["<type>", "null"]`; they stay `required` (optionality is carried by the null type, the right shape for strict structured output).
- **reasoning** — nullable plus an `X-ReasoningPrompt` on each optional field (and an `X-SystemPrompt`) telling the model to return null when a field is absent.
- **enum** — reasoning plus an `enum` of ISO-4217 codes on `currency`, normalizing the printed vocabulary to a canonical code.

Each variant runs every invoice at `n_consensus=5`. Likelihood = mean per-field consensus confidence; weak = below 0.90.

## 1. Corpus summary

| Variant | Overall accuracy | Present-field accuracy | **Absent-field accuracy** | Mean likelihood |
|---|---:|---:|---:|---:|
| baseline | 83% (50/60) | 86% (42/49) | **73%** (8/11) | 0.98 |
| nullable | 90% (54/60) | 88% (43/49) | **100%** (11/11) | 1.00 |
| reasoning | 95% (57/60) | 94% (46/49) | **100%** (11/11) | 0.99 |
| enum | 98% (59/60) | 98% (48/49) | **100%** (11/11) | 0.99 |

The **absent-field** column is the headline: it measures how each variant handles fields that are genuinely missing on an invoice — exactly where a required-everything schema must invent a value.

## 2. Accuracy per invoice

| Invoice | Absent fields | baseline | nullable | reasoning | enum |
|---|---|---:|---:|---:|---:|
| invoice_full | — (all present) | 92% | 92% | 100% | 100% |
| invoice_no_name | customer_name | 83% | 83% | 92% | 100% |
| invoice_no_code | customer_code, purchase_order | 75% | 83% | 92% | 92% |
| invoice_minimal | due_date, customer_name, customer_code, purchase_order, discount, tax | 75% | 92% | 100% | 100% |
| invoice_mixed | purchase_order, discount | 92% | 100% | 92% | 100% |

## 3. What happened on the absent fields

For every field that is *absent* on an invoice, the correct answer is `null`. Here is what each variant actually returned.

| Invoice · field | baseline | nullable | reasoning | enum |
|---|---|---|---|---|
| invoice_no_name · customer_name | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_no_code · customer_code | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_no_code · purchase_order | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · due_date | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · customer_name | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · customer_code | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · purchase_order | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · discount | ✗ `0` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_minimal · tax | ✗ `0` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_mixed · purchase_order | ✓ `None` | ✓ `None` | ✓ `None` | ✓ `None` |
| invoice_mixed · discount | ✗ `0` | ✓ `None` | ✓ `None` | ✓ `None` |

## 4. Currency normalization (the enum lever)

`currency` is present on every invoice but printed in a verbose form; the correct value is the ISO-4217 code. A free-text field echoes the printed vocabulary, an `enum` normalizes it.

| Invoice (→ expected code) | baseline | nullable | reasoning | enum |
|---|---|---|---|---|
| invoice_full (→ `EUR`) | ✓ `'EUR'` | ✓ `'EUR'` | ✓ `'EUR'` | ✓ `'EUR'` |
| invoice_no_name (→ `EUR`) | ✗ `'Euros'` | ✗ `'Euros'` | ✗ `'Euros'` | ✓ `'EUR'` |
| invoice_no_code (→ `USD`) | ✗ `'US Dollars'` | ✗ `'US Dollars'` | ✗ `'US Dollars'` | ✓ `'USD'` |
| invoice_minimal (→ `GBP`) | ✓ `'GBP'` | ✗ `'Pounds Sterling'` | ✓ `'GBP'` | ✓ `'GBP'` |
| invoice_mixed (→ `EUR`) | ✓ `'EUR'` | ✓ `'EUR'` | ✗ `'€'` | ✓ `'EUR'` |

## 5. Stability — consensus agreement

Likelihood measures how strongly the consensus runs **agreed** on a field, not whether the value was right. A field below the threshold (0.90) is **weak** — the runs split. The interesting case is high likelihood on a *wrong* value: the model is confidently wrong, so likelihood cannot be used to rank schemas — only accuracy can.

| Variant | Mean likelihood | Weak fields (< threshold) |
|---|---:|---:|
| baseline | 0.98 | 5 |
| nullable | 1.00 | 1 |
| reasoning | 0.99 | 1 |
| enum | 0.99 | 1 |

Every field that fell below the threshold, with the value returned and whether it was correct:

| Variant | Invoice · field | Likelihood | Returned | Correct |
|---|---|---:|---|:--:|
| baseline | invoice_no_name · customer_name | 0.60 | `None` | ✓ |
| baseline | invoice_no_code · currency | 0.86 | `'US Dollars'` | ✗ |
| baseline | invoice_minimal · due_date | 0.60 | `None` | ✓ |
| baseline | invoice_minimal · currency | 0.86 | `'GBP'` | ✓ |
| baseline | invoice_mixed · currency | 0.65 | `'EUR'` | ✓ |
| nullable | invoice_no_code · currency | 0.86 | `'US Dollars'` | ✗ |
| reasoning | invoice_mixed · currency | 0.65 | `'€'` | ✗ |
| enum | invoice_no_code · discount | 0.60 | `32.25` | ✗ |

Note how few fields are weak even though accuracy varies widely: likelihood stayed near `1.00` across variants while accuracy moved **83% → 98%**. A confident value is not a correct one.


## 4. Interpretation

Three clean lever wins, plus one honest caveat — overall accuracy climbs
**83% → 90% → 95% → 98%**:

- **nullable → absence.** With no discount (and no tax on `minimal`), the
  required non-nullable number had no way to say "absent", so it returned `0`.
  Nullability fixed it (`null`): absent-field accuracy **73% → 100%**. *Mark
  genuinely-optional fields nullable.* (The baseline returned `null` for absent
  *strings* on its own — the reliable failure was on numbers, where `0` looks
  legitimate.)
- **reasoning → sign convention.** The discount is printed negative, yet
  baseline and nullable returned the wrong sign (`150`, `48`, `32.25`) on every
  invoice that had one. The reasoning prompt flipped all three. Present-field
  accuracy **88% → 94%**. *A reasoning prompt encodes a convention a bare type
  can't — but watch the likelihood: in the enum run one discount came back
  positive at confidence `0.60`, the model's own "I'm unsure" flag.*
- **enum → vocabulary.** `currency` is printed verbosely (`Euros`, `US Dollars`,
  `Pounds Sterling`, `€`); a free-text field echoes that, scoring **2–3 / 5**
  against the canonical ISO code. An `enum` of ISO-4217 codes normalized every
  value → **5 / 5**, lifting present-field accuracy **94% → 98%**. *For a
  categorical field, an enum pins output to a fixed vocabulary.*
- **Caveat: the European-date risk is real but noisy.** The baseline misread two
  `DD/MM/YYYY` dates this run (`03/02/2024 → 2024-03-02`); the other variants got
  them right — but the fix already appears at `nullable`, which has no date
  mechanism, so it's partly sampling variance. *A single run can over- or
  under-state a risky pattern — measure across runs.*

**Stability (see §5 of the report above).** Consensus likelihood — how strongly
the five runs agreed — stayed near `1.00` throughout, *including on values the
baseline got wrong*. Only a handful of fields dipped below `0.90`: mostly the
verbose currency on the baseline (the runs were genuinely unsure how to render
it) and the enum-run discount at `0.60`. Weak-field count did fall as the schema
improved (5 → 1), but likelihood barely moved while accuracy climbed 15 points —
so **likelihood measures agreement, not correctness, and cannot rank the
schemas; only accuracy can.**

## 5. The linter

The experiment cost credits to run once. The linter turns its lessons into a
free, instant static check: it reads a schema and flags the same patterns —
no API calls, no documents.


In [4]:
from harness.lint import lint_schema
import json
for name in ["baseline", "nullable", "reasoning", "enum"]:
    schema = json.load(open(f"schemas/{name}.json", encoding="utf-8"))
    findings = lint_schema(schema)
    warns = [f for f in findings if f.severity == "warn"]
    print(f"{name:9s}  {len(warns)} warn")
    for f in warns:
        print(f"    [WARN] {f.path}: {f.rule}")

baseline   3 warn
    [WARN] <schema>: all-required-no-nullable
    [WARN] currency: enum-candidate
    [WARN] discount_amount: sign-convention-no-reason
nullable   2 warn
    [WARN] currency: enum-candidate
    [WARN] discount_amount: sign-convention-no-reason
reasoning  1 warn
    [WARN] currency: enum-candidate
enum       0 warn


### The payoff

The linter's `warn` count tracks the measured accuracy — more warnings, more failures:

| Schema | Linter warns | Measured accuracy |
|---|:--:|:--:|
| baseline | 3 (all-required + discount-sign + currency-enum) | 83% |
| nullable | 2 (discount-sign + currency-enum) | 90% |
| reasoning | 1 (currency-enum) | 95% |
| enum | 0 (clean) | 98% |

The static checks move in lock-step with the live measurements, so the linter
encodes proven patterns rather than guesses. Each warning that clears
corresponds to a measured accuracy gain.

**Takeaways**

1. Mark genuinely-optional fields **nullable** — a required, non-nullable field fabricates values (numbers become `0`). Keep the field `required` but nullable: that's the right shape for strict structured output.
2. Use an **`X-ReasoningPrompt`** to encode conventions a type can't (signs, units, tie-breakers) — and verify it actually lands on your model.
3. Give categorical fields an **`enum`** — a free-text field echoes whatever vocabulary is printed (`Euros`); an enum normalizes it to a canonical value (`EUR`).
4. **Measure** — confidence ≠ correctness, not every risky pattern breaks, and a single run can mislead.
5. Run `python -m harness.lint your_schema.json` before you ship a schema. (`missing-description` is an advisory heuristic — not measured in this folder.)